# Ask AOPWiki a question

Which pathways concern thyroid disruption, and what connects them to genes? Ask the model to investigate AOPWiki and show its evidence. It receives no example query or preselected route.

We open the included schema, not a new mining run. Only the requested records are read from the endpoint.

Install `uv pip install -e ".[agents,mcp,notebooks]"` in your notebook environment. Put `OPENAI_API_KEY` in `notebooks/.env`. `OPENAI_MODEL` is optional; this example uses GPT-5.4 mini by default.

In [1]:
import os
import sys
from dotenv import load_dotenv
from mcp import Client as MCPClient, StdioServerParameters
from pydantic_ai.usage import UsageLimits
from rdfsolve.pydantic_ai import research_agent
from rdfsolve.mcp import read_result
from IPython.display import Markdown, display

load_dotenv("../.env");

## Open the tools

Four tools let the model inspect definitions, search text, choose paths, and read connected records. The session keeps every query and its returned data.

In [5]:
server = MCPClient(
    StdioServerParameters(
        command=sys.executable,
        args=["-m", "rdfsolve.mcp", "--schema", "../data/aopwikirdf.schema.json",
              "--source-id", "aopwikirdf", "--log", "output/investigation-session.json"],
    ),
    read_timeout_seconds=120,
)

## Ask the question

The model chooses where to search and which connections to follow. Its answer should explain what the source supports, not just list matching names.

In [6]:
async with server:
    agent = await research_agent(
        server,
        os.getenv("OPENAI_MODEL", "openai:gpt-5.4-mini-2026-03-17"),
        model_settings={"max_tokens": 2000},
        retries=2,
    )
    answer = await agent.run(
        "Which Adverse Outcome Pathways represent thyroid disruption, and which genes are described to be related to these pathways? What is the evidence for these genes being related?",
        usage_limits=UsageLimits(request_limit=16, tool_calls_limit=20, total_tokens_limit=60000),
    )
    display(Markdown(answer.output.text))
    usage = answer.usage
    tables = [await read_result(server, result.reference) for result in answer.output.results]

for table in tables:
    display(table)

I found several AOPs in AOP-Wiki that explicitly represent thyroid disruption, especially amphibian metamorphosis pathways and one visual-function pathway. The clearest thyroid-disruption AOPs are: AOP 175 (thyroperoxidase/TPO inhibition), AOP 176 (sodium iodide symporter/NIS inhibition), AOP 188 (iodotyrosine deiodinase/IYD inhibition), AOP 189 (DIO1 inhibition), AOP 190 (DIO2 inhibition), AOP 191 (DIO3 inhibition), AOP 192 (pendrin inhibition), AOP 193 (DUOX inhibition), AOP 366 (competitive binding to transthyretin/TTR), AOP 367 (competitive binding to thyroid binding globulin/TBG), and AOP 363 (TPO inhibition leading to altered visual function via retinal layer structure). AOP 194 is also thyroid-axis related (hepatic nuclear receptor activation leading to altered amphibian metamorphosis), but the title is less directly a thyroid-disruption mechanism than the others.

The genes described as related in the retrieved records are DIO1, DIO2, DIO3, DUOX1, DUOX2, and DUOXA1, with support from gene identifiers returned for multiple identifier systems (Ensembl, HGNC, NCBI Gene, UniProt) showing these labels in the AOP-Wiki graph. In the AOP 175 record, the abstract explicitly mentions DIO2 as the major activating enzyme that converts T4 to T3, and the evidence text also discusses thyroid hormone receptor-mediated gene expression and deiodinase-mediated local regulation of T3. The gene identifier search did not return TPO, NIS/SLC5A5, IYD, pendrin/SLC26A4, TTR, or TBG as gene identifiers in the retrieved results, so for those I only have pathway-level evidence from AOP labels/descriptions, not gene-record evidence from this query.

Evidence for the genes being related:
- DIO1/DIO2/DIO3: direct gene-identifier matches in the AOP-Wiki data, plus AOP 175 abstract explicitly names DIO2 as the activating enzyme that converts T4 to T3.
- DUOX1/DUOX2/DUOXA1: direct gene-identifier matches in the AOP-Wiki data, consistent with the DUOX-related thyroid pathway AOP 193.

What counts as evidence here is mainly the AOP-Wiki text and ontology links: the AOP descriptions/abstracts state that these pathways are thyroid disruption mechanisms, and the gene identifier records show the corresponding gene symbols are present in the same knowledge base. I did not retrieve record-level assertion text tying each gene individually to a specific AOP beyond the AOP text itself and the gene identifier matches, so the support is strongest for DIO2 in AOP 175 and for the DUOX/DIO genes as pathway-associated identifiers. There is a coverage limit here because I only retrieved the first page of AOP candidates and a single gene-identifier search page; additional AOPs or related genes may exist in the database beyond what was returned.

,Name,Class,Identifier,Type,label,title
0,Chronic binding of antagonist to N-methyl-D-as...,Adverse Outcome Pathway,https://identifiers.org/aop/12,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 12],[Chronic binding of antagonist to N-methyl-D-a...
1,Chronic binding of antagonist to N-methyl-D-as...,Adverse Outcome Pathway,https://identifiers.org/aop/13,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 13],[Chronic binding of antagonist to N-methyl-D-a...
2,Binding of electrophilic chemicals to SH(thiol...,Adverse Outcome Pathway,https://identifiers.org/aop/17,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 17],[Binding of electrophilic chemicals to SH(thio...
3,Thyroperoxidase inhibition leading to altered ...,Adverse Outcome Pathway,https://identifiers.org/aop/175,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 175],[Thyroperoxidase inhibition leading to altered...
4,Sodium Iodide Symporter (NIS) Inhibition leadi...,Adverse Outcome Pathway,https://identifiers.org/aop/176,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 176],[Sodium Iodide Symporter (NIS) Inhibition lead...
5,Iodotyrosine deiodinase (IYD) inhibition leadi...,Adverse Outcome Pathway,https://identifiers.org/aop/188,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 188],[Iodotyrosine deiodinase (IYD) inhibition lead...
6,Type I iodothyronine deiodinase (DIO1) inhibit...,Adverse Outcome Pathway,https://identifiers.org/aop/189,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 189],[Type I iodothyronine deiodinase (DIO1) inhibi...
7,Type II iodothyronine deiodinase (DIO2) inhibi...,Adverse Outcome Pathway,https://identifiers.org/aop/190,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 190],[Type II iodothyronine deiodinase (DIO2) inhib...
8,Type III iodotyrosine deiodinase (DIO3) inhibi...,Adverse Outcome Pathway,https://identifiers.org/aop/191,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 191],[Type III iodotyrosine deiodinase (DIO3) inhib...
9,Pendrin inhibition leading to altered amphibia...,Adverse Outcome Pathway,https://identifiers.org/aop/192,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[AOP 192],[Pendrin inhibition leading to altered amphibi...


,Name,Class,Identifier,Type,abstract,description,has_evidence,has_key_event,has_molecular_initiating_event,label
0,AOP 175,Adverse Outcome Pathway,https://identifiers.org/aop/175,http://aopkb.org/aop_ontology#AdverseOutcomePa...,[This AOP describes how decreased thyroid horm...,[There is a wealth of information on the inhib...,[Biological plausibility:\n\n&bull; The critic...,"[https://identifiers.org/aop.events/1101, http...",[https://identifiers.org/aop.events/279],[AOP 175]


,Name,Class,Identifier,Type,label
0,DUOX1,Gene identifier,https://identifiers.org/ensembl/ENSG00000137857,http://edamontology.org/data_1025,[DUOX1]
1,DUOXA1,Gene identifier,https://identifiers.org/ensembl/ENSG00000140254,http://edamontology.org/data_1025,[DUOXA1]
2,DUOX2,Gene identifier,https://identifiers.org/ensembl/ENSG00000140279,http://edamontology.org/data_1025,[DUOX2]
3,DIO3,Gene identifier,https://identifiers.org/ensembl/ENSG00000197406,http://edamontology.org/data_1025,[DIO3]
4,DIO2,Gene identifier,https://identifiers.org/ensembl/ENSG00000211448,http://edamontology.org/data_1025,[DIO2]
5,DIO1,Gene identifier,https://identifiers.org/ensembl/ENSG00000211452,http://edamontology.org/data_1025,[DIO1]
6,DUOX2,Gene identifier,https://identifiers.org/hgnc/13273,http://edamontology.org/data_1025,[DUOX2]
7,DUOXA1,Gene identifier,https://identifiers.org/hgnc/26507,http://edamontology.org/data_1025,[DUOXA1]
8,DIO1,Gene identifier,https://identifiers.org/hgnc/2883,http://edamontology.org/data_1025,[DIO1]
9,DIO2,Gene identifier,https://identifiers.org/hgnc/2884,http://edamontology.org/data_1025,[DIO2]


A mention, a connection and experimental evidence are different things. Read the supporting passages and links below to assess the answer.

The explanation is written by the model. The tables contain the actual retrieved records, with their classes and identifiers. Separate result groups stay in separate tables.

Use `tables[0].attrs["records"]` for the first table's generated Python objects. Its `attrs["evidence"]` and `attrs["coverage"]` retain supporting matches and reported limits.

## See what it used

The table lists the tools used. Open an entry below it to see its arguments and answer, or the SPARQL queries and returned rows.

In [7]:
from IPython.display import display
from rdfsolve.query_log import QueryLog

log = QueryLog.read("output/investigation-session.json")
display(log.tools())
display(log)

,Tool,Status,Queries
0,search,failed,[]
1,search,complete,[1]
2,search,complete,"[2, 3, 4]"
3,paths,failed,[]
4,schema,complete,[]
5,search,complete,[5]
6,search,failed,[]
7,search,complete,"[6, 7, 8, 9]"
8,paths,failed,[]
9,read,complete,[10]


s,type,p,text,_graph
s,type,p,text,_graph
https://identifiers.org/aop/12,http://aopkb.org/aop_ontology#AdverseOutcomePathway,http://purl.org/dc/elements/1.1/description,"A prime example of impairments in learning and memory as the adverse outcome for regulatory action is developmental lead exposure and IQ function in children (Bellinger, 2012). Most methods are well established in the published literature and many have been engaged to evaluate the effects of developmental thyroid disruption. The US EPA and OECD Developmental Neurotoxicity (DNT) Guidelines (OCSPP 870.6300 or OECD TG 426) as well as OECD TG 443 (OECD, 2018) both require testing of learning and memory (USEPA, 1998; OECD, 2007) advising to use the following tests passive avoidance, delayed-matching-to-position for the adult rat and for the infant rat, olfactory conditioning, Morris water maze, Biel or Cincinnati maze, radial arm maze, T-maze, and acquisition and retention of schedule-controlled behavior. These DNT Guidelines have been deemed valid to identify developmental neurotoxicity and adverse neurodevelopmental outcomes (Makris et al., 2009).&nbsp;\n\nAlso, in the frame of the OECD GD 43 (2008) on reproductive toxicity, learning and memory testing may have potential to be applied in the context of developmental neurotoxicity studies. However, many of the learning and memory tasks used in guideline studies may not readily detect subtle impairments in cognitive function associated with modest degrees of developmental thyroid disruption (Gilbert et al., 2012).&nbsp;\n\n&nbsp;\n",http://aopwiki.org/
https://identifiers.org/aop/13,http://aopkb.org/aop_ontology#AdverseOutcomePathway,http://purl.org/dc/elements/1.1/description,"A prime example of impairments in learning and memory as the adverse outcome for regulatory action is developmental lead exposure and IQ function in children (Bellinger, 2012). Most methods are well established in the published literature and many have been engaged to evaluate the effects of developmental thyroid disruption. The US EPA and OECD Developmental Neurotoxicity (DNT) Guidelines (OCSPP 870.6300 or OECD TG 426) as well as OECD TG 443 (OECD, 2018) both require testing of learning and memory (USEPA, 1998; OECD, 2007) advising to use the following tests passive avoidance, delayed-matching-to-position for the adult rat and for the infant rat, olfactory conditioning, Morris water maze, Biel or Cincinnati maze, radial arm maze, T-maze, and acquisition and retention of schedule-controlled behavior. These DNT Guidelines have been deemed valid to identify developmental neurotoxicity and adverse neurodevelopmental outcomes (Makris et al., 2009).&nbsp;\n\nAlso, in the frame of the OECD GD 43 (2008) on reproductive toxicity, learning and memory testing may have potential to be applied in the context of developmental neurotoxicity studies. However, many of the learning and memory tasks used in guideline studies may not readily detect subtle impairments in cognitive function associated with modest degrees of developmental thyroid disruption (Gilbert et al., 2012).&nbsp;\n\n&nbsp;\n",http://aopwiki.org/
https://identifiers.org/aop/17,http://aopkb.org/aop_ontology#AdverseOutcomePathway,http://purl.org/dc/elements/1.1/description,"A prime example of impairments in learning and memory as the adverse outcome for regulatory action is developmental lead exposure and IQ function in children (Bellinger, 2012). Most methods are well established in the published literature and many have been engaged to evaluate the effects of developmental thyroid disruption. The US EPA and OECD Developmental Neurotoxicity (DNT) Guidelines (OCSPP 870.6300 or OECD TG 426) as well as OECD TG 443 (OECD, 2018) both require testing of learning and memory (USEPA, 1998; OECD, 2007) advising to use the following tests passive avoidance, delayed-matching-to-position for the adult rat and for the infant rat, olfactory conditioning, Morris water maze, Biel or Cincinnati maze, radial a